In [ ]:
import os
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import jax
import jax.numpy as jnp
import optax
import equinox as eqx
import mlflow
from omegaconf import OmegaConf

from mlflow_sweeper import SweepConfig, run_sweep

from phd.jax_core.tasks.feature_sifting import FeatureSiftingTask
from phd.jax_core.optimizers.idbd import optax_idbd, IDBDState
from phd.jax_core.optimizers.adam import custom_optax_adam, AdamState
from phd.jax_core.utils import tree_replace

from phd.research_utils.scripts.mlflow_download import download_experiment
from phd.research_utils.analysis.analysis_utils import (
    load_experiment_data, get_best_ablation_values, get_color_palette,
)

sns.set_theme('notebook', 'white')
%matplotlib inline

## CIP V1

In [ ]:

def _fable_record_mean(z, r_p, r_w, eps):
    """G(z) = weighted mean over the record R of max(0, P - z). Empty R -> 0."""
    return jnp.sum(r_w * jnp.maximum(r_p - z, 0.0)) / (jnp.sum(r_w) + eps)


def _fable_solve_z_star(r_p, r_w, horizon, tau, n_bisect, eps):
    """The bar z*: it solves ``H * G(z) = TAU * z`` — the value at which the expected
    lifetime winnings (over horizon H) of one fresh redraw past z exactly pay for the
    TAU steps a test burns. phi(z) = H*G(z) - TAU*z is non-increasing with phi(0) >= 0
    and phi(max_P) <= 0, so a fixed-iteration bisection brackets the unique root. An
    empty / zero-mass record (G(0) == 0) means no bar at all, so z* = 0."""
    g0 = _fable_record_mean(0.0, r_p, r_w, eps)
    hi0 = jnp.max(r_p)  # recorded utilities are >= 0, so this upper-bounds the root

    def phi(z):
        return horizon * _fable_record_mean(z, r_p, r_w, eps) - tau * z

    def body(_, bounds):
        lo, hi = bounds
        mid = 0.5 * (lo + hi)
        go_right = phi(mid) > 0.0          # phi decreasing -> root lies to the right
        return (jnp.where(go_right, mid, lo), jnp.where(go_right, hi, mid))

    lo, hi = jax.lax.fori_loop(0, n_bisect, body, (jnp.array(0.0), hi0))
    return jnp.where(g0 <= 0.0, 0.0, 0.5 * (lo + hi))


def _fable_push_records(r_p, r_w, r_ptr, resolved, value, r_lam, r_size):
    """Push each resolved slot's measured quality into the ring buffer R, in slot order.
    Mirrors ``record(P)``: age every existing weight by (1 - r_lam), then write the new
    entry with weight 1.0 at the rolling pointer (overwriting the oldest once full)."""
    def body(j, carry):
        r_p, r_w, r_ptr = carry
        do = resolved[j]
        r_w = jnp.where(do, r_w * (1.0 - r_lam), r_w)          # age all existing entries
        r_p = r_p.at[r_ptr].set(jnp.where(do, value[j], r_p[r_ptr]))
        r_w = r_w.at[r_ptr].set(jnp.where(do, 1.0, r_w[r_ptr]))
        r_ptr = jnp.where(do, (r_ptr + 1) % r_size, r_ptr)
        return (r_p, r_w, r_ptr)

    return jax.lax.fori_loop(0, resolved.shape[0], body, (r_p, r_w, r_ptr))


class Fable(eqx.Module):
    """Fable — reservoir-index pruning. Each candidate slot runs a sequential test of
    "is my potential utility above the bar z*?". A slot is pruned (its task feature
    regenerated) only once its utility *upper* bound falls below z*; the resolved test
    outcomes feed a record R that sets z*. See the header comment for the full loop."""

    # Static hyperparameters (changing any of these recompiles the jitted training loop).
    alpha: float = eqx.field(static=True)        # learner step-size (LMS)
    lam: float = eqx.field(static=True)          # EMA decay; memory ~ 1/lam steps
    tau: float = eqx.field(static=True)          # test cost in steps (pinned to 1/lam)
    horizon: float = eqx.field(static=True)      # H: how many future steps we value
    beta: float = eqx.field(static=True)         # confidence-bound width, in std errors
    timeout_a: float = eqx.field(static=True)    # A: every test resolves within A/lam steps
    r_lam: float = eqx.field(static=True)        # record aging rate
    r_size: int = eqx.field(static=True)         # record capacity
    n_bisect: int = eqx.field(static=True)       # bisection iterations for z*
    eps: float = eqx.field(static=True)

    # Per-feature state (slot j holds one candidate; respawn is also how all slots start).
    v: jax.Array        # weights
    g: jax.Array        # EMA of e*f (doubles as the LMS gradient trace)
    q: jax.Array        # EMA of (e*f)^2
    m: jax.Array        # EMA of f*f (starts at 0.25)
    age: jax.Array
    counted: jax.Array  # bool: has this draw's quality been recorded into R yet?

    # The record R of resolved-test qualities, as a weighted ring buffer.
    r_p: jax.Array      # recorded utilities
    r_w: jax.Array      # their (aged) weights
    r_ptr: jax.Array    # next write index

    prune_mask: jax.Array  # slots the task should regenerate next step (<=1 True)

    @classmethod
    def init(cls, hparams, key):
        n = N_LEARNER_FEATURES
        lam = hparams['lam']
        tau = hparams.get('tau')
        tau = (1.0 / lam) if tau is None else tau
        r_size = int(hparams.get('r_size', 128))
        return cls(
            alpha=hparams['learning_rate'],
            lam=lam,
            tau=tau,
            horizon=hparams['horizon'],
            beta=hparams.get('beta', 2.0),
            timeout_a=hparams.get('timeout_a', 4.0),
            r_lam=hparams.get('r_lam', 0.01),
            r_size=r_size,
            n_bisect=int(hparams.get('n_bisect', 40)),
            eps=hparams.get('eps', 1e-8),
            v=jnp.zeros(n), g=jnp.zeros(n), q=jnp.zeros(n),
            m=jnp.full(n, 0.25), age=jnp.zeros(n),
            counted=jnp.zeros(n, dtype=bool),
            r_p=jnp.zeros(r_size), r_w=jnp.zeros(r_size), r_ptr=jnp.array(0, dtype=jnp.int32),
            prune_mask=jnp.zeros(n, dtype=bool),
        )

    def step(self, x, y):
        f = x                                   # each slot's frozen feature definition
        e = y - jnp.sum(self.v * f)             # prediction error (full residual)
        lam = self.lam

        # 1. learn and measure ------------------------------------------------------
        v = self.v + self.alpha * e * f         # LMS update; g below is its gradient trace
        ef = e * f
        g = (1.0 - lam) * self.g + lam * ef
        m = (1.0 - lam) * self.m + lam * (f * f)
        q = (1.0 - lam) * self.q + lam * ef ** 2
        age = self.age + 1.0

        # 2. potential utility with confidence bounds -------------------------------
        n_eff = jnp.minimum(age, 2.0 / lam)
        se = self.beta * jnp.sqrt(jnp.maximum(q - g * g, 0.0) / n_eff)
        a = g + v * m                           # effective target for this slot
        lo_b, hi_b = a - se, a + se
        p = a * a / (m + self.eps)              # point estimate of utility
        ucb = jnp.maximum(lo_b ** 2, hi_b ** 2) / (m + self.eps)
        lcb = jnp.where(lo_b * hi_b > 0.0,      # 0 when the interval straddles 0
                        jnp.minimum(lo_b ** 2, hi_b ** 2), 0.0) / (m + self.eps)

        # 3. the bar z* (from the record R as it stands before this step's records) -
        z_star = _fable_solve_z_star(self.r_p, self.r_w, self.horizon, self.tau,
                                     self.n_bisect, self.eps)

        # 4. record tests that just resolved ----------------------------------------
        not_counted = ~self.counted
        pass_test = not_counted & (lcb > z_star)                 # confirmed above bar -> p
        fail_test = not_counted & (ucb < z_star)                 # censored below bar -> ucb
        timeout = (not_counted & ~pass_test & ~fail_test         # ran out of patience -> p
                   & (age > self.timeout_a / lam))
        resolved = pass_test | fail_test | timeout
        record_value = jnp.where(pass_test, p, jnp.where(fail_test, ucb, p))
        counted = self.counted | resolved
        r_p, r_w, r_ptr = _fable_push_records(
            self.r_p, self.r_w, self.r_ptr, resolved, record_value, self.r_lam, self.r_size)

        # 5. prune at most one feature per step (utilities are coupled) --------------
        j = jnp.argmin(ucb)
        do_prune = ucb[j] < z_star
        prune_mask = jnp.zeros_like(self.prune_mask).at[j].set(do_prune)

        # respawn the pruned slot: the task regenerates its feature next step, and we
        # reset its state. m carries over the mean scale (refines within ~1/lam steps).
        v = jnp.where(prune_mask, 0.0, v)
        g = jnp.where(prune_mask, 0.0, g)
        q = jnp.where(prune_mask, 0.0, q)
        age = jnp.where(prune_mask, 0.0, age)
        m = jnp.where(prune_mask, jnp.mean(m), m)
        counted = jnp.where(prune_mask, False, counted)

        return tree_replace(self, v=v, g=g, q=q, m=m, age=age, counted=counted,
                            r_p=r_p, r_w=r_w, r_ptr=r_ptr, prune_mask=prune_mask), e ** 2


FABLE_DEFAULTS = {
    'learning_rate': 0.05,   # ALPHA: LMS step-size
    'lam': 0.005,            # EMA decay (memory ~ 1/lam = 200 steps); tau defaults to 1/lam
    'horizon': 100_000.0,    # H: the real knob — how many future steps a redraw is valued over
    'beta': 2.0,             # confidence-bound width, in standard errors
    'timeout_a': 4.0,        # every test resolves within A/lam = 800 steps
    'r_lam': 0.01,           # record aging rate
    'r_size': 128,           # record capacity
    'n_bisect': 40,          # bisection iterations for z*
}
METHODS['fable'] = MethodSpec('fable', 'Fable', Fable, FABLE_DEFAULTS)

## CIP V1 — numpy port

A plain-numpy version of the JAX `Fable` above: same algorithm, with the
jit-compatibility scaffolding stripped out. The `fori_loop` record-pushing
becomes a Python list, `_solve_z_star`'s bisection is a plain `for` loop, and
state is mutated in place — no `eqx.Module` / `tree_replace`.

We run it once with the simple training-loop format from
`building_new_method.ipynb` and plot some statistics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from phd.jax_core.tasks.feature_sifting import FeatureSiftingTaskNP as FeatureSiftingTask


class Fable:
    """Reservoir-index pruning by sequential confidence-bound tests (numpy).

    Per feature we keep a weight ``v``, three EMAs (``g`` of e*f, ``m`` of f*f,
    ``q`` of (e*f)^2), an ``age``, and one ``counted`` bit. Globally we keep a
    small weighted record ``R`` of how good resolved feature-tests turned out.

    Every step, identically: learn -> measure -> bound the potential utility ->
    compute the bar ``z*`` from R -> record any tests that just resolved -> prune
    at most the single worst feature whose utility *upper* bound is below the bar.
    A pruned slot is respawned (the task regenerates its feature next step).

    Cold start needs no special handling: an empty R gives z* = 0 (nothing is
    pruned) until the timeout has fed enough resolved tests into R for the bar to
    lift off. Young features survive because a small effective sample count gives
    them wide error bars and therefore a large utility upper bound.
    """

    def __init__(
        self,
        n_features,
        alpha=0.05,        # LMS step size
        lam=0.005,         # EMA decay; memory ~ 1/lam steps
        horizon=1000.0,    # H: how many future steps a fresh redraw is valued over (the main knob)
        tau=None,          # test cost in steps (defaults to 1/lam)
        beta=2.0,          # confidence-bound width, in standard errors
        timeout_a=4.0,     # every test resolves within timeout_a / lam steps
        r_lam=0.01,        # record aging rate
        r_size=128,        # record capacity
        eps=1e-8,
    ):
        self.n = n_features
        self.alpha = alpha
        self.lam = lam
        self.horizon = horizon
        self.tau = (1.0 / lam) if tau is None else tau
        self.beta = beta
        self.timeout_a = timeout_a
        self.r_lam = r_lam
        self.r_size = r_size
        self.eps = eps

        # Per-feature state.
        self.v = np.zeros(n_features)
        self.g = np.zeros(n_features)
        self.q = np.zeros(n_features)
        self.m = np.full(n_features, 0.25)
        self.age = np.zeros(n_features)
        self.counted = np.zeros(n_features, dtype=bool)

        # The record R of resolved-test utilities: parallel lists of values and
        # (aged) weights, trimmed to r_size. No ring buffer / pointer arithmetic.
        self.r_p = []
        self.r_w = []

        # Exposed for logging.
        self.prune_mask = np.zeros(n_features, dtype=bool)
        self.z_star = 0.0

    def _record_mean(self, z):
        """G(z) = weighted mean over R of max(0, P - z). Empty R -> 0."""
        if not self.r_p:
            return 0.0
        r_p, r_w = np.asarray(self.r_p), np.asarray(self.r_w)
        return np.sum(r_w * np.maximum(r_p - z, 0.0)) / (np.sum(r_w) + self.eps)

    def _solve_z_star(self):
        """The bar z* solves H*G(z) = tau*z. phi(z) = H*G(z) - tau*z is
        non-increasing with phi(0) >= 0, so a plain bisection finds the unique
        root. An empty / zero-mass record means no bar (z* = 0)."""
        if self._record_mean(0.0) <= 0.0:
            return 0.0
        lo, hi = 0.0, max(self.r_p)
        for _ in range(40):
            mid = 0.5 * (lo + hi)
            if self.horizon * self._record_mean(mid) - self.tau * mid > 0.0:
                lo = mid           # phi decreasing -> root is to the right
            else:
                hi = mid
        return 0.5 * (lo + hi)

    def _record(self, value):
        """Age every existing weight by (1 - r_lam), append (value, weight=1),
        and drop the oldest entry once we exceed capacity."""
        self.r_w = [w * (1.0 - self.r_lam) for w in self.r_w]
        self.r_p.append(float(value))
        self.r_w.append(1.0)
        if len(self.r_p) > self.r_size:
            self.r_p.pop(0)
            self.r_w.pop(0)

    def step(self, x, y):
        f = x
        e = y - np.sum(self.v * f)
        lam = self.lam

        # 1. learn and measure
        self.v += self.alpha * e * f
        ef = e * f
        self.g = (1 - lam) * self.g + lam * ef
        self.m = (1 - lam) * self.m + lam * (f * f)
        self.q = (1 - lam) * self.q + lam * ef ** 2
        self.age += 1.0

        # 2. potential utility with confidence bounds
        n_eff = np.minimum(self.age, 2.0 / lam)
        se = self.beta * np.sqrt(np.maximum(self.q - self.g ** 2, 0.0) / n_eff)
        a = self.g + self.v * self.m
        lo_b, hi_b = a - se, a + se
        p = a ** 2 / (self.m + self.eps)
        ucb = np.maximum(lo_b ** 2, hi_b ** 2) / (self.m + self.eps)
        lcb = np.where(lo_b * hi_b > 0.0,
                       np.minimum(lo_b ** 2, hi_b ** 2), 0.0) / (self.m + self.eps)

        # 3. the bar z* (from R as it stands before this step's records)
        self.z_star = self._solve_z_star()

        # 4. record tests that just resolved
        for j in np.where(~self.counted)[0]:
            if lcb[j] > self.z_star:                        # confirmed above bar -> p
                self._record(p[j]); self.counted[j] = True
            elif ucb[j] < self.z_star:                      # censored below bar -> ucb
                self._record(ucb[j]); self.counted[j] = True
            elif self.age[j] > self.timeout_a / lam:         # timeout -> p
                self._record(p[j]); self.counted[j] = True

        # 5. prune at most one feature per step (utilities are coupled)
        self.prune_mask = np.zeros(self.n, dtype=bool)
        j = int(np.argmin(ucb))
        if ucb[j] < self.z_star:
            self.prune_mask[j] = True
            self.v[j] = self.g[j] = self.q[j] = 0.0
            self.age[j] = 0.0
            self.m[j] = np.mean(self.m)        # scale carry-over; refines within 1/lam
            self.counted[j] = False

        return e ** 2

    def utility(self):
        """Current point-estimate utility p per feature (for plotting / analysis)."""
        a = self.g + self.v * self.m
        return a ** 2 / (self.m + self.eps)

In [ ]:
### Setup ###
SEED = 0
INPUT_DIM = 20

### Task init ###
task = FeatureSiftingTask(n_learner_features=INPUT_DIM, seed=SEED)
x, y = task.step(np.zeros(INPUT_DIM))

### Parameters ###
train_steps = 20000
log_freq = 2000

### Method init ###
fable = Fable(n_features=INPUT_DIM, horizon=1000.0)

### Stats to track ###
losses, z_stars, cum_prunes = [], [], []
total_prunes = 0

for i in range(train_steps):

    ### Step (forward pass + LMS update + pruning decision) ###
    loss = fable.step(x, y)

    ### Track stats ###
    losses.append(loss)
    z_stars.append(fable.z_star)
    total_prunes += int(fable.prune_mask.sum())
    cum_prunes.append(total_prunes)

    ### New sample (the task regenerates any pruned slots) ###
    x, y = task.step(fable.prune_mask)

    ### Logging ###
    if (i + 1) % log_freq == 0:
        print(f'Step {i + 1:>6} | loss {np.mean(losses[-log_freq:]):.4f} | '
              f'z* {fable.z_star:.4f} | prunes {total_prunes}')

losses = np.array(losses)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

### Smoothed learning curve ###
ax = axes[0, 0]
window = 200
smooth = np.convolve(losses, np.ones(window) / window, mode='valid')
ax.plot(np.arange(window - 1, train_steps), smooth)
ax.set_xlabel('Step'); ax.set_ylabel(f'Loss ({window}-step MA)')
ax.set_title('Learning curve'); ax.set_yscale('log'); ax.grid(True)

### The bar z* over time ###
ax = axes[0, 1]
ax.plot(z_stars)
ax.set_xlabel('Step'); ax.set_ylabel('z*')
ax.set_title('The bar z* (lifts off once R has mass)'); ax.grid(True)

### Cumulative prunes ###
ax = axes[1, 0]
ax.plot(cum_prunes)
ax.set_xlabel('Step'); ax.set_ylabel('Cumulative prunes')
ax.set_title(f'Pruning activity ({total_prunes} total)'); ax.grid(True)

### Final per-feature utility vs noise coefficient ###
ax = axes[1, 1]
ax.scatter(task.noise_coefficients, fable.utility())
ax.set_xlabel('Feature noise coefficient'); ax.set_ylabel('Final utility (p)')
ax.set_title('Surviving features: utility vs noise'); ax.grid(True)

plt.tight_layout()
plt.show()

print(f'Final loss (last 2k steps): {losses[-2000:].mean():.4f}')
print(f'corr(noise coefficient, utility): '
      f'{np.corrcoef(task.noise_coefficients, fable.utility())[0, 1]:.3f}')